# Multi-Resolution Ensemble Piano Transcriber - Training

Trains a lightweight (~770K param) Conv1d + BiGRU meta-learner that fuses 373 GPU-parallel
spectro-temporal features into per-key onset/frame/velocity predictions.

**Features (computed on GPU in one pass):**
- 3 mel spectrograms at different STFT resolutions (n_fft=1024/2048/4096) = 264
- CQT via filterbank = 88
- Chromagram = 12
- 9 onset detection functions (spectral flux, RMS, HFC x 3 resolutions)

**After training:** Download `ensemble_transcription.pt` (~5MB) and place it in
`backend/rhythm_training/` on your local machine.

In [ ]:
!pip install -q pretty_midi librosa

In [ ]:
import csv
import json
import os
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Find MAESTRO Dataset

In [ ]:
# Auto-detect MAESTRO location on Kaggle
KAGGLE_INPUT = Path("/kaggle/input")
MAESTRO_DIR = None

# Search common Kaggle dataset paths
for candidate in [
    KAGGLE_INPUT / "maestro-v300" / "maestro-v3.0.0",
    KAGGLE_INPUT / "maestro-v300",
    KAGGLE_INPUT / "maestro" / "maestro-v3.0.0",
    KAGGLE_INPUT / "maestro",
]:
    csv_path = candidate / "maestro-v3.0.0.csv"
    if csv_path.exists():
        MAESTRO_DIR = candidate
        break

if MAESTRO_DIR is None:
    # List what's actually in /kaggle/input
    print("Could not auto-detect MAESTRO. Contents of /kaggle/input:")
    for p in sorted(KAGGLE_INPUT.rglob("*.csv")):
        print(f"  {p}")
    raise FileNotFoundError("Set MAESTRO_DIR manually to the folder containing maestro-v3.0.0.csv")

MAESTRO_CSV = MAESTRO_DIR / "maestro-v3.0.0.csv"
print(f"MAESTRO_DIR: {MAESTRO_DIR}")
print(f"CSV: {MAESTRO_CSV}")

# Count audio files
wav_files = list(MAESTRO_DIR.rglob("*.wav"))
midi_files = list(MAESTRO_DIR.rglob("*.midi")) + list(MAESTRO_DIR.rglob("*.mid"))
print(f"WAV files: {len(wav_files)}")
print(f"MIDI files: {len(midi_files)}")

## 2. Constants

In [ ]:
SAMPLE_RATE = 16000
HOP_LENGTH = 512
N_MELS = 88
PIANO_KEYS = 88
MIDI_OFFSET = 21
N_FFTS = [1024, 2048, 4096]
CQT_BINS = 88
CHROMA_BINS = 12
ONSET_FEATURES = 9
N_FEATURES = N_MELS * len(N_FFTS) + CQT_BINS + CHROMA_BINS + ONSET_FEATURES  # 373

SEGMENT_SECONDS = 10.0
SEGMENT_FRAMES = int(SEGMENT_SECONDS * SAMPLE_RATE / HOP_LENGTH)

OUTPUT_DIR = Path("/kaggle/working")
INDEX_DIR = OUTPUT_DIR / "ensemble_index"
MODEL_PATH = OUTPUT_DIR / "ensemble_transcription.pt"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Features per frame: {N_FEATURES}")
print(f"Segment frames: {SEGMENT_FRAMES}")
print(f"Device: {DEVICE}")

## 3. Multi-Resolution Feature Extractor

In [ ]:
class MultiResFeatureExtractor:
    """
    GPU-accelerated multi-resolution feature extraction.
    Computes 373 features per frame from audio in a single GPU pass.
    """

    def __init__(self, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                 n_mels=N_MELS, n_ffts=None, device=None):
        self.sr = sr
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.n_ffts = n_ffts or N_FFTS
        self.device = device or DEVICE

        # Precompute STFT windows
        self.windows = {}
        for n_fft in self.n_ffts:
            self.windows[n_fft] = torch.hann_window(n_fft, device=self.device)

        # Precompute mel filterbanks
        import librosa
        self.mel_fbs = {}
        for n_fft in self.n_ffts:
            fb = librosa.filters.mel(
                sr=self.sr, n_fft=n_fft, n_mels=self.n_mels,
                fmin=27.5, fmax=self.sr // 2,
            )
            self.mel_fbs[n_fft] = torch.from_numpy(fb.astype(np.float32)).to(self.device)

        # Precompute CQT filterbank
        self.cqt_fb = self._build_cqt_filterbank(max(self.n_ffts))

    def _build_cqt_filterbank(self, n_fft):
        bins_per_octave = 12
        fmin = 27.5
        Q = 1.0 / (2.0 ** (1.0 / bins_per_octave) - 1.0)
        freqs = np.fft.rfftfreq(n_fft, 1.0 / self.sr)
        filterbank = np.zeros((CQT_BINS, len(freqs)), dtype=np.float32)
        for k in range(CQT_BINS):
            f_center = fmin * (2.0 ** (k / bins_per_octave))
            bw = f_center / Q
            sigma = bw / 4.0
            if sigma < 1e-6:
                continue
            weights = np.exp(-0.5 * ((freqs - f_center) / sigma) ** 2)
            weights[freqs < f_center - 3 * sigma] = 0
            weights[freqs > f_center + 3 * sigma] = 0
            w_sum = np.sum(weights)
            if w_sum > 0:
                filterbank[k] = weights / w_sum
        return torch.from_numpy(filterbank).to(self.device)

    def extract(self, audio):
        """Extract 373 features. audio: (batch, samples) or (samples,) tensor."""
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)
        audio = audio.to(self.device)

        magnitudes = {}
        for n_fft in self.n_ffts:
            stft = torch.stft(
                audio, n_fft, hop_length=self.hop_length,
                window=self.windows[n_fft], return_complex=True, center=True,
            )
            magnitudes[n_fft] = torch.abs(stft)

        n_frames = min(m.size(-1) for m in magnitudes.values())
        for n_fft in self.n_ffts:
            magnitudes[n_fft] = magnitudes[n_fft][:, :, :n_frames]

        parts = []

        # 1. Mel spectrograms (3 x 88 = 264)
        for n_fft in self.n_ffts:
            mel = torch.matmul(self.mel_fbs[n_fft].unsqueeze(0), magnitudes[n_fft])
            mel = torch.log(mel + 1e-6)
            parts.append(mel)

        # 2. CQT (88)
        largest_mag = magnitudes[max(self.n_ffts)]
        cqt = torch.matmul(self.cqt_fb.unsqueeze(0), largest_mag)
        cqt_log = torch.log(cqt + 1e-6)
        parts.append(cqt_log)

        # 3. Chromagram (12)
        batch_size = cqt.size(0)
        cqt_padded = F.pad(cqt, (0, 0, 0, 96 - CQT_BINS))
        chroma = cqt_padded.view(batch_size, 8, 12, n_frames).sum(dim=1)
        chroma = chroma / (chroma.sum(dim=1, keepdim=True) + 1e-8)
        parts.append(chroma)

        # 4. Onset functions (9)
        for n_fft in self.n_ffts:
            mag = magnitudes[n_fft]

            diff = torch.diff(mag, dim=-1)
            diff = torch.clamp(diff, min=0)
            flux = diff.sum(dim=1)
            flux = F.pad(flux, (1, 0))
            flux = flux / (flux.max(dim=-1, keepdim=True).values + 1e-8)
            parts.append(flux.unsqueeze(1))

            rms = torch.sqrt(torch.mean(mag ** 2, dim=1))
            rms = rms / (rms.max(dim=-1, keepdim=True).values + 1e-8)
            parts.append(rms.unsqueeze(1))

            freq_weights = torch.linspace(0, 1, mag.size(1), device=self.device)
            hfc = (mag ** 2 * freq_weights.view(1, -1, 1)).sum(dim=1)
            hfc = torch.sqrt(hfc)
            hfc = hfc / (hfc.max(dim=-1, keepdim=True).values + 1e-8)
            parts.append(hfc.unsqueeze(1))

        all_features = torch.cat(parts, dim=1)
        return all_features.permute(0, 2, 1)

    @property
    def n_features(self):
        return N_FEATURES

# Quick test
extractor = MultiResFeatureExtractor(device=DEVICE)
test_audio = torch.randn(1, SAMPLE_RATE * 2, device=DEVICE)  # 2 seconds
with torch.no_grad():
    test_feats = extractor.extract(test_audio)
print(f"Feature shape for 2s audio: {test_feats.shape}  (expected: [1, ~63, 373])")
del test_audio, test_feats
torch.cuda.empty_cache()

## 4. Meta-Learner Model

In [ ]:
class EnsembleMetaLearner(nn.Module):
    """
    Conv1d + BiGRU meta-learner (~770K params).
    3x Conv1d -> 2-layer BiGRU -> onset/frame/velocity heads.
    """

    def __init__(self, n_features=N_FEATURES, conv_channels=None,
                 gru_hidden=64, gru_layers=2, n_keys=PIANO_KEYS, dropout=0.1):
        super().__init__()
        if conv_channels is None:
            conv_channels = [256, 256, 128]

        self.n_features = n_features
        self.n_keys = n_keys

        self.conv1 = nn.Conv1d(n_features, conv_channels[0], kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(conv_channels[0])
        self.conv2 = nn.Conv1d(conv_channels[0], conv_channels[1], kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(conv_channels[1])
        self.conv3 = nn.Conv1d(conv_channels[1], conv_channels[2], kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(conv_channels[2])

        self.gru = nn.GRU(
            conv_channels[2], gru_hidden, num_layers=gru_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if gru_layers > 1 else 0,
        )

        gru_out_dim = gru_hidden * 2

        self.onset_head = nn.Sequential(
            nn.Linear(gru_out_dim, gru_out_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(gru_out_dim, n_keys),
        )
        self.frame_head = nn.Sequential(
            nn.Linear(gru_out_dim, gru_out_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(gru_out_dim, n_keys),
        )
        self.velocity_head = nn.Sequential(
            nn.Linear(gru_out_dim, gru_out_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(gru_out_dim, n_keys), nn.Sigmoid(),
        )

    def forward(self, x):
        h = x.permute(0, 2, 1)
        h = F.gelu(self.bn1(self.conv1(h)))
        h = F.gelu(self.bn2(self.conv2(h)))
        h = F.gelu(self.bn3(self.conv3(h)))
        h = h.permute(0, 2, 1)
        h, _ = self.gru(h)
        return {
            'onset_logits': self.onset_head(h),
            'frame_logits': self.frame_head(h),
            'velocity': self.velocity_head(h),
        }


model = EnsembleMetaLearner().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## 5. Loss Function

In [ ]:
class EnsembleLoss(nn.Module):
    """
    Velocity-weighted loss: soft notes penalized 2-3x more when missed.
    weight = 1 + alpha * (1 - velocity), alpha=2.0
    """

    def __init__(self, alpha=2.0, pos_weight=5.0,
                 onset_weight=1.0, frame_weight=1.0, velocity_weight=0.5):
        super().__init__()
        self.alpha = alpha
        self.pos_weight = pos_weight
        self.onset_w = onset_weight
        self.frame_w = frame_weight
        self.vel_w = velocity_weight

    def forward(self, onset_logits, frame_logits, velocity_pred,
                onset_gt, frame_gt, velocity_gt):
        vel_weight = torch.ones_like(velocity_gt)
        active = frame_gt > 0.5
        if active.any():
            vel_weight[active] = 1.0 + self.alpha * (1.0 - velocity_gt[active])

        onset_bce = F.binary_cross_entropy_with_logits(
            onset_logits, onset_gt, reduction='none')
        onset_sample_w = torch.where(
            onset_gt > 0.5, vel_weight * self.pos_weight, torch.ones_like(vel_weight))
        onset_loss = (onset_bce * onset_sample_w).mean()

        frame_bce = F.binary_cross_entropy_with_logits(
            frame_logits, frame_gt, reduction='none')
        frame_sample_w = torch.where(
            frame_gt > 0.5, vel_weight * self.pos_weight, torch.ones_like(vel_weight))
        frame_loss = (frame_bce * frame_sample_w).mean()

        if active.any():
            velocity_loss = F.mse_loss(velocity_pred[active], velocity_gt[active])
        else:
            velocity_loss = torch.tensor(0.0, device=onset_logits.device)

        total = self.onset_w * onset_loss + self.frame_w * frame_loss + self.vel_w * velocity_loss
        return {'total': total, 'onset': onset_loss, 'frame': frame_loss, 'velocity': velocity_loss}

## 6. Dataset

In [ ]:
class CachedEnsembleDataset(Dataset):
    """
    Dataset that loads audio on-demand via librosa.
    Slightly slower than pre-cached .npy but avoids disk space issues.
    """

    def __init__(self, index_path, sr=SAMPLE_RATE, hop_length=HOP_LENGTH):
        self.sr = sr
        self.hop_length = hop_length
        self.segment_frames = SEGMENT_FRAMES
        self.segment_samples = self.segment_frames * hop_length

        with open(index_path) as f:
            self.index = json.load(f)
        self.segments = self.index['segments']
        self.pieces = self.index['pieces']

        # Store audio paths (no pre-loading to save RAM/disk)
        self.audio_paths = {}
        for i, piece in enumerate(self.pieces):
            # Use cached path if .npy exists, otherwise use original audio path
            cache_path = piece.get('audio_cached', '')
            if cache_path and cache_path.endswith('.npy') and os.path.exists(cache_path):
                self.audio_paths[i] = ('npy', cache_path)
            else:
                self.audio_paths[i] = ('wav', piece['audio'])

        print(f"[Dataset] {len(self.segments)} segments from {len(self.pieces)} pieces (on-demand loading)")

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        seg = self.segments[idx]
        piece_idx = seg['piece_idx']
        start_sec = seg['start_sec']
        start_sample = int(start_sec * self.sr)

        audio_type, audio_path = self.audio_paths.get(piece_idx, ('wav', self.pieces[piece_idx]['audio']))

        if audio_type == 'npy':
            # Fast path: load from cached numpy array
            audio_arr = np.load(audio_path)
            end_sample = start_sample + self.segment_samples
            audio = audio_arr[start_sample:end_sample].copy()
            if len(audio) < self.segment_samples:
                audio = np.pad(audio, (0, self.segment_samples - len(audio)))
        else:
            # Load directly via librosa (on-demand)
            import librosa
            audio, _ = librosa.load(
                audio_path, sr=self.sr, mono=True,
                offset=start_sec, duration=SEGMENT_SECONDS,
            )
            if len(audio) < self.segment_samples:
                audio = np.pad(audio, (0, self.segment_samples - len(audio)))
            audio = audio[:self.segment_samples]

        onset, frame, velocity = self._create_labels(
            self.pieces[piece_idx]['midi'], start_sec)
        return {
            'audio': torch.from_numpy(audio).float(),
            'onset': onset, 'frame': frame, 'velocity': velocity,
        }

    def _create_labels(self, midi_path, start_sec):
        import pretty_midi
        midi = pretty_midi.PrettyMIDI(midi_path)

        onset = np.zeros((self.segment_frames, PIANO_KEYS), dtype=np.float32)
        frame = np.zeros((self.segment_frames, PIANO_KEYS), dtype=np.float32)
        velocity = np.zeros((self.segment_frames, PIANO_KEYS), dtype=np.float32)

        end_sec = start_sec + SEGMENT_SECONDS
        frame_time = self.hop_length / self.sr

        for instrument in midi.instruments:
            if instrument.is_drum:
                continue
            for note in instrument.notes:
                if note.end < start_sec or note.start > end_sec:
                    continue
                key = note.pitch - MIDI_OFFSET
                if key < 0 or key >= PIANO_KEYS:
                    continue
                onset_f = int((note.start - start_sec) / frame_time)
                offset_f = int((note.end - start_sec) / frame_time)
                onset_f = max(0, min(onset_f, self.segment_frames - 1))
                offset_f = max(0, min(offset_f, self.segment_frames))
                vel_norm = note.velocity / 127.0
                for f in range(onset_f, min(onset_f + 2, self.segment_frames)):
                    onset[f, key] = 1.0
                for f in range(onset_f, offset_f):
                    frame[f, key] = 1.0
                    velocity[f, key] = vel_norm

        return torch.from_numpy(onset), torch.from_numpy(frame), torch.from_numpy(velocity)

## 7. Prepare Segment Index

In [ ]:
import librosa

all_pieces = []
with open(MAESTRO_CSV, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        all_pieces.append(row)
print(f"Total pieces in CSV: {len(all_pieces)}")

# Check audio availability
pieces_with_audio = []
for piece in all_pieces:
    audio_path = MAESTRO_DIR / piece['audio_filename']
    midi_path = MAESTRO_DIR / piece['midi_filename']
    if audio_path.exists() and midi_path.exists():
        pieces_with_audio.append(piece)

print(f"Pieces with audio+MIDI: {len(pieces_with_audio)}")

if len(pieces_with_audio) == 0:
    raise RuntimeError("No audio files found! Make sure the MAESTRO dataset includes WAV files.")

# Build index per split
INDEX_DIR.mkdir(parents=True, exist_ok=True)

for split in ['train', 'validation', 'test']:
    split_pieces = [p for p in pieces_with_audio if p['split'] == split]
    pieces_list = []
    segments = []

    for piece in split_pieces:
        audio_path = str(MAESTRO_DIR / piece['audio_filename'])
        midi_path = str(MAESTRO_DIR / piece['midi_filename'])
        try:
            duration = librosa.get_duration(path=audio_path)
        except Exception as e:
            print(f"  Skipping {audio_path}: {e}")
            continue

        stored_idx = len(pieces_list)
        pieces_list.append({
            'audio': audio_path,
            'midi': midi_path,
            'composer': piece.get('canonical_composer', ''),
            'title': piece.get('canonical_title', ''),
            'duration': duration,
        })
        for start in np.arange(0, duration - SEGMENT_SECONDS / 2, SEGMENT_SECONDS):
            segments.append({'piece_idx': stored_idx, 'start_sec': float(start)})

    index = {
        'pieces': pieces_list,
        'segments': segments,
        'sr': SAMPLE_RATE,
        'hop_length': HOP_LENGTH,
        'segment_seconds': SEGMENT_SECONDS,
    }
    index_path = INDEX_DIR / f"{split}_index.json"
    with open(index_path, 'w') as f:
        json.dump(index, f)
    print(f"  {split}: {len(pieces_list)} pieces, {len(segments)} segments")

print(f"\nIndex saved to {INDEX_DIR}")

## 7.5 Audio Path Setup (no pre-caching)

Maps audio paths for on-demand loading. This avoids Kaggle disk space limits (~20GB) since caching all MAESTRO audio would require ~40GB.

In [ ]:
# Skip pre-caching to avoid disk space issues on Kaggle
# Audio will be loaded on-demand via librosa (fast enough, feature extraction is the bottleneck)

# Collect all unique audio paths from all splits
all_audio_paths = set()
for split in ['train', 'validation', 'test']:
    index_path = INDEX_DIR / f"{split}_index.json"
    if index_path.exists():
        with open(index_path) as f:
            idx = json.load(f)
        for piece in idx['pieces']:
            all_audio_paths.add(piece['audio'])

print(f"Total audio files: {len(all_audio_paths)}")

# Map each audio path to itself (no caching, load directly)
audio_cache_map = {p: p for p in sorted(all_audio_paths)}

# Save the cache mapping (points to original files)
AUDIO_CACHE_DIR = INDEX_DIR  # Reuse index dir for the map file
cache_map_path = AUDIO_CACHE_DIR / "cache_map.json"
with open(cache_map_path, 'w') as f:
    json.dump(audio_cache_map, f)

# Update index files with audio_cached pointing to original files
for split in ['train', 'validation', 'test']:
    index_path = INDEX_DIR / f"{split}_index.json"
    if index_path.exists():
        with open(index_path) as f:
            idx = json.load(f)
        for piece in idx['pieces']:
            piece['audio_cached'] = piece['audio']  # Use original path
        with open(index_path, 'w') as f:
            json.dump(idx, f)

print("Using on-demand audio loading (no pre-caching, saves ~40GB disk space)")

## 8. Train

In [ ]:
# ─── Hyperparameters ───
EPOCHS = 25
BATCH_SIZE = 16
LR = 3e-4
GRU_HIDDEN = 64
GRU_LAYERS = 2
DROPOUT = 0.1
VEL_ALPHA = 2.0
POS_WEIGHT = 5.0
NUM_WORKERS = 2  # Kaggle has limited CPU cores

In [ ]:
# Feature extractor
extractor = MultiResFeatureExtractor(device=DEVICE)
print(f"Feature extractor: {extractor.n_features} features per frame")

# Datasets (using cached .npy files — no librosa I/O during training)
train_dataset = CachedEnsembleDataset(str(INDEX_DIR / "train_index.json"))
val_dataset = CachedEnsembleDataset(str(INDEX_DIR / "validation_index.json"))

# NUM_WORKERS=0: all data comes from RAM, no disk I/O needed
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True,
)

# Model
conv_channels = [256, 256, 128]
model = EnsembleMetaLearner(
    n_features=extractor.n_features,
    conv_channels=conv_channels,
    gru_hidden=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Loss, optimizer, scheduler
criterion = EnsembleLoss(alpha=VEL_ALPHA, pos_weight=POS_WEIGHT)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# ─── Training Loop (with AMP mixed precision for T4 speedup) ───
best_val_loss = float('inf')
best_onset_f1 = 0.0
history = []

# AMP: mixed precision for ~2-3x speedup on T4 tensor cores
use_amp = DEVICE.type == 'cuda'
scaler = torch.amp.GradScaler(enabled=use_amp)

for epoch in range(EPOCHS):
    t0 = time.time()

    # ── Train ──
    model.train()
    train_losses = defaultdict(float)
    n_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        audio = batch['audio'].to(DEVICE)
        onset_gt = batch['onset'].to(DEVICE)
        frame_gt = batch['frame'].to(DEVICE)
        vel_gt = batch['velocity'].to(DEVICE)

        with torch.no_grad():
            features = extractor.extract(audio)

        T = min(features.size(1), onset_gt.size(1))
        features = features[:, :T, :]
        onset_gt = onset_gt[:, :T, :]
        frame_gt = frame_gt[:, :T, :]
        vel_gt = vel_gt[:, :T, :]

        optimizer.zero_grad()

        # AMP: forward pass in float16
        with torch.amp.autocast('cuda', enabled=use_amp):
            out = model(features)
            losses = criterion(
                out['onset_logits'], out['frame_logits'], out['velocity'],
                onset_gt, frame_gt, vel_gt,
            )

        # AMP: scaled backward + optimizer step
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        for k, v in losses.items():
            train_losses[k] += v.item()
        n_batches += 1

        if batch_idx % 50 == 0 and batch_idx > 0:
            avg = train_losses['total'] / n_batches
            print(f"  Epoch {epoch+1} batch {batch_idx}/{len(train_loader)}: loss={avg:.4f}")

    scheduler.step()

    # ── Validate ──
    model.eval()
    val_losses = defaultdict(float)
    n_val = 0
    onset_tp = onset_fp = onset_fn = 0
    frame_tp = frame_fp = frame_fn = 0

    with torch.no_grad():
        for batch in val_loader:
            audio = batch['audio'].to(DEVICE)
            onset_gt = batch['onset'].to(DEVICE)
            frame_gt = batch['frame'].to(DEVICE)
            vel_gt = batch['velocity'].to(DEVICE)

            features = extractor.extract(audio)
            T = min(features.size(1), onset_gt.size(1))
            features = features[:, :T, :]
            onset_gt = onset_gt[:, :T, :]
            frame_gt = frame_gt[:, :T, :]
            vel_gt = vel_gt[:, :T, :]

            # AMP: validation forward in float16 too
            with torch.amp.autocast('cuda', enabled=use_amp):
                out = model(features)
                losses = criterion(
                    out['onset_logits'], out['frame_logits'], out['velocity'],
                    onset_gt, frame_gt, vel_gt,
                )
            for k, v in losses.items():
                val_losses[k] += v.item()
            n_val += 1

            onset_pred = (torch.sigmoid(out['onset_logits']) > 0.5).float()
            frame_pred = (torch.sigmoid(out['frame_logits']) > 0.5).float()

            onset_tp += ((onset_pred == 1) & (onset_gt == 1)).sum().item()
            onset_fp += ((onset_pred == 1) & (onset_gt == 0)).sum().item()
            onset_fn += ((onset_pred == 0) & (onset_gt == 1)).sum().item()

            frame_tp += ((frame_pred == 1) & (frame_gt == 1)).sum().item()
            frame_fp += ((frame_pred == 1) & (frame_gt == 0)).sum().item()
            frame_fn += ((frame_pred == 0) & (frame_gt == 1)).sum().item()

    onset_p = onset_tp / max(onset_tp + onset_fp, 1)
    onset_r = onset_tp / max(onset_tp + onset_fn, 1)
    onset_f1 = 2 * onset_p * onset_r / max(onset_p + onset_r, 1e-8)
    frame_p = frame_tp / max(frame_tp + frame_fp, 1)
    frame_r = frame_tp / max(frame_tp + frame_fn, 1)
    frame_f1 = 2 * frame_p * frame_r / max(frame_p + frame_r, 1e-8)

    avg_train = {k: v / max(n_batches, 1) for k, v in train_losses.items()}
    avg_val = {k: v / max(n_val, 1) for k, v in val_losses.items()}
    elapsed = time.time() - t0

    print(f"\nEpoch {epoch+1}/{EPOCHS} ({elapsed:.0f}s)")
    print(f"  Train loss: {avg_train['total']:.4f} "
          f"(onset={avg_train['onset']:.4f}, frame={avg_train['frame']:.4f}, "
          f"vel={avg_train['velocity']:.4f})")
    print(f"  Val loss:   {avg_val['total']:.4f}")
    print(f"  Onset  P={onset_p:.3f} R={onset_r:.3f} F1={onset_f1:.3f}")
    print(f"  Frame  P={frame_p:.3f} R={frame_r:.3f} F1={frame_f1:.3f}")

    history.append({
        'epoch': epoch + 1, 'train_loss': avg_train['total'],
        'val_loss': avg_val['total'], 'onset_f1': onset_f1, 'frame_f1': frame_f1,
    })

    if avg_val['total'] < best_val_loss:
        best_val_loss = avg_val['total']
        best_onset_f1 = onset_f1
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': {
                'n_features': extractor.n_features,
                'conv_channels': conv_channels,
                'gru_hidden': GRU_HIDDEN,
                'gru_layers': GRU_LAYERS,
                'n_keys': PIANO_KEYS,
                'sample_rate': SAMPLE_RATE,
                'hop_length': HOP_LENGTH,
            },
            'epoch': epoch,
            'val_loss': best_val_loss,
            'onset_f1': onset_f1,
            'frame_f1': frame_f1,
        }, str(MODEL_PATH))
        print(f"  ** Saved best model! (val_loss={best_val_loss:.4f}, onset_f1={onset_f1:.3f})")

print(f"\n{'='*60}")
print(f"Training complete!")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"  Best onset F1: {best_onset_f1:.3f}")
print(f"  Checkpoint: {MODEL_PATH}")
print(f"  Size: {MODEL_PATH.stat().st_size / 1e6:.1f} MB")

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_list = [h['epoch'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_list, [h['train_loss'] for h in history], label='Train')
ax1.plot(epochs_list, [h['val_loss'] for h in history], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_list, [h['onset_f1'] for h in history], label='Onset F1')
ax2.plot(epochs_list, [h['frame_f1'] for h in history], label='Frame F1')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')
ax2.set_title('Onset & Frame F1')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()

## 10. Verify Checkpoint

Quick sanity check: load the saved checkpoint and run inference on a real audio clip.

In [ ]:
# Load checkpoint
checkpoint = torch.load(str(MODEL_PATH), map_location=DEVICE, weights_only=False)
config = checkpoint['config']
print(f"Checkpoint config: {config}")
print(f"Trained for {checkpoint['epoch']+1} epochs")
print(f"Val loss: {checkpoint['val_loss']:.4f}")
print(f"Onset F1: {checkpoint['onset_f1']:.3f}")
print(f"Frame F1: {checkpoint['frame_f1']:.3f}")

# Load model
test_model = EnsembleMetaLearner(
    n_features=config['n_features'],
    conv_channels=config['conv_channels'],
    gru_hidden=config['gru_hidden'],
    gru_layers=config['gru_layers'],
    n_keys=config['n_keys'],
).to(DEVICE)
test_model.load_state_dict(checkpoint['model_state_dict'])
test_model.eval()

# Grab a real audio file for testing
test_piece = pieces_with_audio[0]
test_audio_path = str(MAESTRO_DIR / test_piece['audio_filename'])
print(f"\nTest piece: {test_piece.get('canonical_title', 'unknown')}")
print(f"  by {test_piece.get('canonical_composer', 'unknown')}")

test_audio, _ = librosa.load(test_audio_path, sr=SAMPLE_RATE, mono=True, duration=30.0)
audio_t = torch.from_numpy(test_audio).float().to(DEVICE)

test_extractor = MultiResFeatureExtractor(device=DEVICE)

with torch.no_grad():
    features = test_extractor.extract(audio_t)
    out = test_model(features)

onset_probs = torch.sigmoid(out['onset_logits'][0]).cpu().numpy()
frame_probs = torch.sigmoid(out['frame_logits'][0]).cpu().numpy()
velocity = out['velocity'][0].cpu().numpy()

# Count detected notes
onset_detected = (onset_probs > 0.4).sum()
frame_active = (frame_probs > 0.3).sum()
print(f"\nInference on 30s clip:")
print(f"  Feature shape: {features.shape}")
print(f"  Onset detections (>0.4): {onset_detected}")
print(f"  Active frames (>0.3): {frame_active}")
print(f"  Max onset prob: {onset_probs.max():.3f}")
print(f"  Max frame prob: {frame_probs.max():.3f}")

del test_model, test_extractor
torch.cuda.empty_cache()

## 11. Download Checkpoint

The checkpoint is at `/kaggle/working/ensemble_transcription.pt`.

To use it locally, download it and place it at:
```
LiveScore/backend/rhythm_training/ensemble_transcription.pt
```

It will be automatically loaded by the pipeline on next run.

In [ ]:
print(f"Checkpoint path: {MODEL_PATH}")
print(f"Checkpoint size: {MODEL_PATH.stat().st_size / 1e6:.1f} MB")
print(f"\nDownload this file and place it at:")
print(f"  LiveScore/backend/rhythm_training/ensemble_transcription.pt")